# 01 - Ingestão: CSV para Delta Lake (MinIO)

In [ ]:
# ==============================================================================
# 1. Configuração e Conexão com o MinIO
# ==============================================================================
# Este bloco define as credenciais de acesso ao Object Storage (MinIO)
# que foi iniciado através do Docker Compose no ambiente local.

import os
import boto3
from botocore.config import Config
from pyspark.sql import SparkSession

MINIO_ENDPOINT = "http://127.0.0.1:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "miniopass"
BUCKET_DESTINO = "datalake"

# Cria o bucket se ainda não existir, porque o MinIO precisa do bucket antes da escrita Delta.
s3_client = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1",
    config=Config(s3={"addressing_style": "path"}),
)

try:
    s3_client.head_bucket(Bucket=BUCKET_DESTINO)
    print(f"Bucket '{BUCKET_DESTINO}' já existe.")
except Exception:
    s3_client.create_bucket(Bucket=BUCKET_DESTINO)
    print(f"Bucket '{BUCKET_DESTINO}' criado com sucesso.")

print(f"Endpoint configurado: {MINIO_ENDPOINT}")
print(f"Bucket de destino: {BUCKET_DESTINO}")

## 2. Inicialização da SparkSession com suporte a Delta e S3

In [ ]:
# ==============================================================================
# 2. Inicialização da SparkSession com suporte a Delta e S3
# ==============================================================================
# Configuramos a sessão do Spark injetando os pacotes do Hadoop-AWS e
# ativando as extensões do Delta Lake para permitir escrita em Object Storage.

spark = SparkSession.builder \
    .appName("Trabalho2-CSV-para-Delta") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

print("Spark conectado ao MinIO com sucesso!")

## 3. Processo de Ingestão e Carga dos Dados

In [ ]:
# ==============================================================================
# 3. Processo de Ingestão e Carga dos Dados (Corrigido para Inglês)
# ==============================================================================

# Agora com os nomes exatos dos ficheiros que o seu colega exportou
tabelas = ["students", "subjects", "classes", "grades", "teacher"]

for tabela in tabelas:
    # Caminho corrigido para a pasta correta: export/data/
    caminho_csv = f"export/data/{tabela}.csv"
    caminho_delta = f"s3a://{BUCKET_DESTINO}/bronze/{tabela}"
    
    print(f"A processar a tabela: {tabela}...")
    
    try:
        # Leitura do ficheiro CSV
        df = spark.read.csv(caminho_csv, header=True, inferSchema=True)
        
        # Escrita no Data Lake em formato Delta
        df.write.format("delta").mode("overwrite").save(caminho_delta)
        
        print(f" -> {tabela} guardada com sucesso no formato Delta!")
    except Exception as e:
        print(f" -> Erro ao processar a tabela {tabela}: {e}")

print("\nProcesso de Ingestão concluído!")